# Building Multi-Agent Systems with CrewAI

A minimal, working CrewAI setup that runs on **Groq** (free, fast inference).

This notebook builds a small two-agent _crew_ — a **Researcher** that gathers key
points on a topic, and a **Writer** that turns those points into a short brief.

**How to run:** use _Run All_ (top toolbar) so every cell executes top-to-bottom
in order. Running a single cell in isolation will fail because later cells depend
on variables defined earlier.


## 1. Install dependencies

`crewai` pulls in `litellm` (the layer that talks to Groq) and everything else.
`python-dotenv` loads your API key from a local `.env` file.


In [1]:
%pip install -q crewai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load the API key and configure the LLM

Create a file named `.env` next to this notebook containing:

```
GROQ_API_KEY=your_real_key_here
```

Get a free key at https://console.groq.com/keys. The `.env` file is git-ignored,
so your key never gets committed.


In [2]:
from dotenv import load_dotenv
import os
from crewai import LLM

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file next to this notebook "
        "with a line like:  GROQ_API_KEY=your_key_here"
    )

# Groq models are addressed as "groq/<model-name>" via LiteLLM.
llm = LLM(model="groq/llama-3.1-8b-instant", api_key=api_key)

print("LLM configured:", llm.model)

21:09:05 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [Errno 11001] getaddrinfo failed. Falling back to local backup.


LLM configured: groq/llama-3.1-8b-instant


## 3. Compatibility shim for Groq

CrewAI 1.15.x tags every message with a `cache_breakpoint` field — a prompt-caching
hint that only Anthropic's API understands. When you route through Groq, that extra
field is sent verbatim and Groq's strict schema rejects it:

> `property 'cache_breakpoint' is unsupported`

The line below neutralizes the tagging so the same code works on any provider.
Remove it once you switch to Anthropic or once CrewAI fixes this upstream.


In [3]:
import crewai.llms.cache as _crewai_cache

# Make the cache-breakpoint tag a no-op so non-Anthropic providers accept the request.
_crewai_cache.mark_cache_breakpoint = lambda message: message

print("Groq compatibility shim applied.")

Groq compatibility shim applied.


## 4. Define the agents

Each agent has a **role**, a **goal**, and a **backstory** that shapes how it behaves.
Both agents share the same `llm` we configured above.


In [4]:
from crewai import Agent

researcher = Agent(
    role="Research Analyst",
    goal="Find and summarize the most important, current facts about {topic}",
    backstory=(
        "You are a meticulous analyst who distills noisy information into a short "
        "list of accurate, high-signal bullet points."
    ),
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Technical Writer",
    goal="Turn research notes about {topic} into a clear, engaging short brief",
    backstory=(
        "You are a writer who turns dense notes into crisp prose that a busy "
        "reader can absorb in under a minute."
    ),
    llm=llm,
    verbose=True,
)

## 5. Define the tasks

Tasks describe _what_ to do and the _expected output_. The `{topic}` placeholder is
filled in at run time from the `inputs` dict. The writer's task lists the research
task in `context`, so the researcher's output is passed to the writer automatically.


In [5]:
from crewai import Task

research_task = Task(
    description="Research {topic} and list the key points a newcomer should know.",
    expected_output="4-6 concise bullet points, each one fact or insight.",
    agent=researcher,
)

writing_task = Task(
    description="Using the research notes, write a short brief on {topic}.",
    expected_output="A 2-3 paragraph brief in plain, engaging language.",
    agent=writer,
    context=[research_task],
)

## 6. Assemble and run the crew

`Process.sequential` runs tasks in order: research first, then writing.

Jupyter already runs an asyncio event loop, so we use `await crew.kickoff_async(...)`
instead of the synchronous `crew.kickoff(...)` (which raises inside a running loop).


In [6]:
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async(inputs={"topic": "multi-agent AI systems"})

print()
print("=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0076d71c-fa63-4419-a91a-cb58e3d408c3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research multi-agent AI systems and list the key points a newcomer should know.                          │
│  ID: 5416e5d9-83da-4364-bcf8-c36b9c5d75fd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: Research multi-agent AI systems and list the key points a newcomer should know.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Key Points for Multi-Agent AI Systems:**                                                                     │
│                                                                                                                 │
│  • **Definition and Scope:** Multi-agent AI systems are a type of artificial intelligence where multiple        │
│  software agents interact and make decisions within a shared environment. These agents can be either            │
│  independent or interconnected, and can be found in real-world applications such as robotics, autonomous        │
│  vehicles, and video games.                                                                                     │
│                                                                                                                 │
│  • **Types of Multi-Agent Systems:** There are several types of multi-agent systems, including:                 │
│    - **Single-level multi-agent systems:** where all agents operate at the same level.                          │
│    - **Multi-level multi-agent systems:** where agents operate at different levels, and there is a hierarchy    │
│  between them.                                                                                                  │
│    - **Hierarchical multi-agent systems:** where agents are organized in a hierarchical structure.              │
│    - **Decentralized multi-agent systems:** where agents make decisions without a central controller.           │
│                                                                                                                 │
│  • **Agent Architecture:** Multi-agent systems often use one of the following types of agent architectures:     │
│    - **Belief-Desire-Intention (BDI) architecture:** This architecture is based on an agent's beliefs,          │
│  desires, and intentions to make decisions.                                                                     │
│    - **BDI-based cognitive architectures:** This architecture is a more complex variation of the BDI            │
│  architecture.                                                                                                  │
│    - **Cognitive architectures with cognitive maps:** This architecture uses cognitive maps to help agents      │
│  understand their environment and make decisions.                                                               │
│                                                                                                                 │
│  • **Communication and Coordination:** Communication and coordination are crucial in multi-agent systems, as    │
│  agents need to share information and coordinate their actions to achieve a common goal. Common methods for     │
│  communication and coordination include:                                                                        │
│    - **Agent-based systems with explicit communication protocols:** This involves agents explicitly             │
│  communicating with each other through predefined protocols.                                                    │
│    - **Stigmergy:** This involves agents modifying their environment, which in turn affects other agents.       │
│    - **Decentralized optimization:** This involves agents optimizing their own objectives, while still working  │
│  towards a common goal.                                

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Multi-Agent AI Systems: Unlocking Collaborative Intelligence**                                               │
│                                                                                                                 │
│  In today's complex, interconnected world, Multi-Agent AI systems are revolutionizing the way we design and     │
│  deploy intelligent systems. By simulating the interactions of multiple software agents, these systems enable   │
│  collaboration, coordination, and decision-making within a shared environment. Found in a wide range of         │
│  applications, including robotics, autonomous vehicles, and video games, Multi-Agent AI systems have the        │
│  potential to transform industries and create new experiences.                                                  │
│                                                                                                                 │
│  At the heart of these systems lie different types of Multi-Agent Systems, each with its own strengths and      │
│  weaknesses. Single-level and multi-level Multi-Agent Systems operate at different levels, while Hierarchical   │
│  and Decentralized Multi-Agent Systems boast organized and self-managed structures. Agent Architectures, such   │
│  as the Belief-Desire-Intention (BDI) architecture, play a crucial role in decision-making, with various        │
│  architectures, including BDI-based cognitive architectures and cognitive architectures with cognitive maps,    │
│  providing different approaches to understanding the environment.                                               │
│                                                                                                                 │
│  Effective communication and coordination are essential for Multi-Agent Systems to achieve a common goal. This  │
│  is achieved through various methods, including Agent-based systems with explicit communication protocols,      │
│  Stigmergy, and Decentralized optimization. As these systems continue to evolve, we're seeing exciting          │
│  applications in Autonomous vehicles, Robotics, Video games, and Smart cities. But with these developments      │
│  come new challenges, including Scalability, Autonomy, Complexity, and Ethics. Researchers are working          │
│  tirelessly to address these issues and unlock the full potential of Multi-Agent AI systems, paving the way     │
│  for a future of collaborative intelligence and innovation.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research notes, write a short brief on multi-agent AI systems.                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0076d71c-fa63-4419-a91a-cb58e3d408c3                                                                       │
│  Final Output: **Multi-Agent AI Systems: Unlocking Collaborative Intelligence**                                 │
│                                                                                                                 │
│  In today's complex, interconnected world, Multi-Agent AI systems are revolutionizing the way we design and     │
│  deploy intelligent systems. By simulating the interactions of multiple software agents, these systems enable   │
│  collaboration, coordination, and decision-making within a shared environment. Found in a wide range of         │
│  applications, including robotics, autonomous vehicles, and video games, Multi-Agent AI systems have the        │
│  potential to transform industries and create new experiences.                                                  │
│                                                                                                                 │
│  At the heart of these systems lie different types of Multi-Agent Systems, each with its own strengths and      │
│  weaknesses. Single-level and multi-level Multi-Agent Systems operate at different levels, while Hierarchical   │
│  and Decentralized Multi-Agent Systems boast organized and self-managed structures. Agent Architectures, such   │
│  as the Belief-Desire-Intention (BDI) architecture, play a crucial role in decision-making, with various        │
│  architectures, including BDI-based cognitive architectures and cognitive architectures with cognitive maps,    │
│  providing different approaches to understanding the environment.                                               │
│                                                                                                                 │
│  Effective communication and coordination are essential for Multi-Agent Systems to achieve a common goal. This  │
│  is achieved through various methods, including Agent-based systems with explicit communication protocols,      │
│  Stigmergy, and Decentralized optimization. As these systems continue to evolve, we're seeing exciting          │
│  applications in Autonomous vehicles, Robotics, Video games, and Smart cities. But with these developments      │
│  come new challenges, including Scalability, Autonomy, Complexity, and Ethics. Researchers are working          │
│  tirelessly to address these issues and unlock the full potential of Multi-Agent AI systems, paving the way     │
│  for a future of collaborative intelligence and innovation.                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
**Multi-Agent AI Systems: Unlocking Collaborative Intelligence**

In today's complex, interconnected world, Multi-Agent AI systems are revolutionizing the way we design and deploy intelligent systems. By simulating the interactions of multiple software agents, these systems enable collaboration, coordination, and decision-making within a shared environment. Found in a wide range of applications, including robotics, autonomous vehicles, and video games, Multi-Agent AI systems have the potential to transform industries and create new experiences.

At the heart of these systems lie different types of Multi-Agent Systems, each with its own strengths and weaknesses. Single-level and multi-level Multi-Agent Systems operate at different levels, while Hierarchical and Decentralized Multi-Agent Systems boast organized and self-managed structures. Agent Architectures, such as the Belief-Desire-Intention (BDI) architecture, play a crucial role in decision-making, with various archit

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 7. Inspect the run

`result` is a `CrewOutput` object. You can pull out each task's individual output
and the token usage for the whole run.


In [7]:
print("Token usage:", result.token_usage)
print()
for i, task_output in enumerate(result.tasks_output, start=1):
    print(f"--- Task {i}: {task_output.agent} ---")
    print(task_output.raw)
    print()

Token usage: total_tokens=3528 prompt_tokens=1740 cached_prompt_tokens=0 completion_tokens=1788 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=4

--- Task 1: Research Analyst ---
**Key Points for Multi-Agent AI Systems:**

• **Definition and Scope:** Multi-agent AI systems are a type of artificial intelligence where multiple software agents interact and make decisions within a shared environment. These agents can be either independent or interconnected, and can be found in real-world applications such as robotics, autonomous vehicles, and video games.

• **Types of Multi-Agent Systems:** There are several types of multi-agent systems, including:
  - **Single-level multi-agent systems:** where all agents operate at the same level.
  - **Multi-level multi-agent systems:** where agents operate at different levels, and there is a hierarchy between them.
  - **Hierarchical multi-agent systems:** where agents are organized in a hierarchical structure.
  - **Decentralized mult